### Step 1: Import API keys and libraries

In [2]:
import os
import json
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API_KEY is missing.")

/workspaces/AIchallenge/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 2: Simple RAG

In [3]:
from document_chunker import chunk_document

with open("document.txt") as f:
    text = f.read()

chunks = chunk_document(text, chunk_size=800, chunk_overlap=100,
                        metadata={"source": "your_document.txt"})

print(f"{len(chunks)} chunks")
chunks[0]  # inspect the first one

47 chunks


Chunk(text='Netflix Culture\n\nEntertainment, like friendship, is a fundamental human need; it changes how we feel and gives us common ground. We want to entertain the world. If we succeed, there is more laughter, more empathy, and more joy. \n\nTo get there, we have an amazing and unusual employee culture. This document is about that culture. \n\nLike all great companies, we strive to hire the best and we value integrity, excellence, respect, inclusion, and collaboration. What is special about Netflix, though, is how much we:\n\n', index=0, start_char=0, end_char=516, metadata={'source': 'your_document.txt', 'chunk_index': 0})

In [10]:
def display_chunks_md(chunks, n=None):
    """Render chunks as markdown. If n is given, only show the first n."""
    subset = chunks[:n] if n is not None else chunks
    for c in subset:
        md = (
            f"### Chunk {c.index}  \n"
            f"*chars {c.start_char}–{c.end_char} · ~{c.token_estimate} tokens*  \n"
            f"*metadata:* `{c.metadata}`\n\n"
            f"> {c.text.strip().replace(chr(10), chr(10) + '> ')}\n\n"
            f"---"
        )
        display(Markdown(md))
    if n is not None and n < len(chunks):
        display(Markdown(f"*...{len(chunks) - n} more chunks not shown*"))

# display_chunks_md(chunks, n=5)
from pprint import pprint
pprint(chunks)


[Chunk(text='Netflix Culture\n'
            '\n'
            'Entertainment, like friendship, is a fundamental human need; it '
            'changes how we feel and gives us common ground. We want to '
            'entertain the world. If we succeed, there is more laughter, more '
            'empathy, and more joy. \n'
            '\n'
            'To get there, we have an amazing and unusual employee culture. '
            'This document is about that culture. \n'
            '\n'
            'Like all great companies, we strive to hire the best and we value '
            'integrity, excellence, respect, inclusion, and collaboration. '
            'What is special about Netflix, though, is how much we:\n'
            '\n',
       index=0,
       start_char=0,
       end_char=516,
       metadata={'chunk_index': 0, 'source': 'your_document.txt'}),
 Chunk(text='nce, respect, inclusion, and collaboration. What is special about '
            'Netflix, though, is how much we:\n'
         

### Step 3: Chunking

In [ ]:
# Generate embeddings
client = OpenAI()

chunks_just_text = just_text = [c.text for c in chunks]


response = client.embeddings.create(
    model="text-embedding-3-small",
    input=chunks_just_text
)

### Step 4: Create Embeddings

In [ ]:
embeddings = [item.embedding for item in response.data]

pprint(response.data)

print(f"Generated {len(embeddings)}")
print(f"Each embedding has {len(embeddings[0])} dimensions")

### Step 5: Initialize ChromaDB and Store Vectors

In [ ]:
import chromadb
chroma_client =  chromadb.PersistentClient(path="./chroma_db")

In [23]:
collection = chroma_client.get_or_create_collection(name="netflix_culture")

if collection.get()["ids"]:
    collection.delete(collection.get()["ids"])

pprint(collection.get())

{'data': None,
 'documents': [],
 'embeddings': None,
 'ids': [],
 'included': ['metadatas', 'documents'],
 'metadatas': [],
 'uris': None}


In [ ]:
ids=[f"chunk_{i}" for i in range(len(chunks_just_text))]
metadatas = [{"source": "netflix_culture_pdf", "chunk_index": i} for i in range(len(chunks_just_text))]

collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=chunks_just_text,
    metadatas=metadatas
)

pprint(collection.get())

# embeddings will be None because chromadb leaves them out by default to "keep it lightweight"

### Step 6: Test Retrieval

In [32]:
test_query = "days off"
test_query2 = "teamwork and collaboration"

response = client.embeddings.create(
    model = "text-embedding-3-small",
    input = [test_query, test_query2]
)

query_embedding = [response.data[0].embedding, response.data[1].embedding]

# Search chromadb
results = collection.query(
    query_embeddings=query_embedding,
    n_results=3
)

print(f"Query: {test_query}")
print("Retrieved Chunks:")
for a, b in zip(results["documents"], results["metadatas"][0]):
    print(f"Chunk {b['chunk_index']}:\n{a}\n")

Query: days off
Retrieved Chunks:
Chunk 28:
['s expected to seek advice and perspective as appropriate. “Use good judgment” is our core precept. \nOur policy for travel, entertainment, gifts, and other expenses is 5 words long: “act in Netflix’s best interest.” \nOur vacation policy is “take vacation.” We don’t have any rules or forms around how many weeks per year. Frankly, we intermix work and personal time quite a bit, doing email at odd hours, taking off a weekday afternoon, etc. Our leaders make sure they set good examples by taking vacations, often coming back with fresh ideas, and encourage the rest of the team to do the same. \nOur parental leave policy is: “take care of your baby and yourself.” New parents generally take 4-8 months. \n', 'neffective. We are always on guard if too much error prevention hinders inventive, creative work. \n\nOn rare occasion, freedom is abused. We had one senior employee who organized kickbacks on IT contracts for example. But those are the excep

### Step 7: Function to Process the Conversation Turn (with RAG)

In [34]:
system_message = "You are a helpful assistant. that answers questions based on the provided context.\
    If you don't know the answer, just say that you don't know.  Always use all available information\
        to provide the best answer possible."

In [ ]:
def respond_ai(message, history):
    # RAG
    response = client.embeddings.create(
        model = "text-embedding-3-small",
        input = [message]
    )

    query_embedding = response.data[0].embedding
    
    # Search chromadb
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )
    
    context = "\n---\n".join(results["documents"][0])
    system_message_enhanced = system_message + "\n\nContext: " + context
    
    messages = [{"role": "system", "content": system_message_enhanced + "***"}]  + history + [{"role": "user", "content": message}]    
    
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    
    reply = response.choices[0].message.content
    return reply

In [ ]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True, inline=False)